# 2026-09-10 — celltype x direction x module per-stage log2FC heatmaps

指定 **celltype x direction x module** 的单独热图，风格与 `2026-09-10_celltype_direction_log2FC_heatmap` 完全一致。

本批 3 张：
1. Central – MG – Ion & solute transport
2. Peripheral – Cone – Synaptic signaling & neurotransmission
3. Central – AC – Synaptic signaling & neurotransmission

- 行：该 celltype-direction **指定 module** 内的 anchor 交集基因。
- 列：`Early, Mid, Late | Young, Old`（胚胎 | 成年，中间小分隔）；**整列无 log2FC 的 stage 不画**。
- 值：DESeq2 每期 **有符号 log2FoldChange**，不做 z-score。
- 行按 **subclass x module** 分块，块内聚类并显示聚类树枝。
- topAnnotation：Cell type + Stage。leftAnnotation：Subclass + Module。图例写全名。
- 配色沿用 `color.ipynb` + module 固定表。

In [ ]:
suppressPackageStartupMessages({
  library(dplyr); library(tidyr); library(readr); library(stringr)
  library(ComplexHeatmap); library(circlize); library(grid)
})

date_str <- "2026-09-10"
data_dir <- "/project/work/2026-09-01_GSEA_test/Data/"
plot_dir <- "/project/work/2026-09-01_GSEA_test/Plot/"

matrix_csv      <- file.path(data_dir, "2026-09-10_celltype_direction_log2FC_matrix.csv")
module_col_file <- file.path(data_dir, "2026-09-06_module_colors.csv")

celltype_order <- c("Rod", "Cone", "AC", "BC", "HC", "MG")
fate_order     <- c("DM", "DAL", "AAM")
stage_order    <- c("Early", "Mid", "Late", "Young", "Old")
period_order   <- c("Embryonic", "Adult")
period_of_stage <- c(Early="Embryonic", Mid="Embryonic", Late="Embryonic", Young="Adult", Old="Adult")

## 配色：来自 color.ipynb
cols.celltype <- c('#A6CEE3','#1F78B4','#B2DF8A','#33A02C','#FB9A99','#E31A1C')
names(cols.celltype) <- celltype_order
cols.stage.full <- c(Early='#59A14F', Mid='#EDC948', Late='#B07AA1', Young='#4E79A7', Old='#E15759')
cols.subclass.short <- c(DM='#66C2A5', DAL='#FC8D62', AAM='#8DA0CB')  # 2026-10-19 更新；旧版 DM=#3C5488, DAL=#E64B35, AAM='#00A087'

## module 固定配色
mod_col_tbl <- read_csv(module_col_file, show_col_types = FALSE)
cols.module <- setNames(mod_col_tbl$color, mod_col_tbl$module)

cat("配色就绪。module 数:", length(cols.module), "\n")

In [ ]:
# ---- 指定 celltype x direction x module 组合 -----------------------------
combos <- tibble::tribble(
  ~direction,      ~celltype, ~module,
  "Central",       "MG",      "Ion & solute transport",
  "Central",       "MG",      "Small-molecule metabolism",
  "Central",       "AC",      "Synaptic signaling & neurotransmission",
  "Peripheral",    "MG",      "Ion & solute transport",
  "Central",       "BC",      "Small-molecule metabolism",
  "Peripheral",    "BC",      "Organ development & morphogenesis"
)

stage_lfc_cols <- paste0(stage_order, "_log2FoldChange")
matrix_long <- read_csv(matrix_csv, show_col_types = FALSE)
cat("矩阵 CSV 行数:", nrow(matrix_long), "\n")
combos

In [ ]:
# ---- 构建 + 绘制单 module 热图 ----------------------------------------------
present_stages <- function(m) {
  keep <- sapply(stage_lfc_cols, function(c) any(!is.na(m[[c]])))
  stage_order[keep]
}

make_module_heatmap <- function(one_direction, one_ct, one_module) {
  m <- matrix_long %>%
    filter(direction == one_direction, celltype == one_ct, module == one_module) %>%
    distinct(subclass, module, gene, .keep_all = TRUE)

  if (nrow(m) == 0) return(NULL)

  stages_keep <- present_stages(m)
  cols_keep   <- paste0(stages_keep, "_log2FoldChange")

  m <- m %>%
    mutate(subclass = factor(subclass, levels = fate_order),
           module   = factor(module)) %>%
    arrange(subclass, module)

  m <- m %>% mutate(module = factor(as.character(module), levels = one_module))

  block_chr <- as.character(m$subclass)
  m$block   <- factor(block_chr, levels = intersect(fate_order, unique(block_chr)))

  mat <- as.matrix(m[, cols_keep])
  colnames(mat) <- stages_keep
  rownames(mat) <- make.unique(m$gene)

  top_anno <- HeatmapAnnotation(
    `Cell type` = factor(rep(one_ct, length(stages_keep)), levels = one_ct),
    Stage       = factor(stages_keep, levels = stages_keep),
    col = list(`Cell type` = cols.celltype, Stage = cols.stage.full),
    show_annotation_name = FALSE, show_legend = TRUE
  )
  left_anno <- rowAnnotation(
    Subclass = m$subclass,
    Module   = m$module,
    col = list(Subclass = cols.subclass.short, Module = cols.module),
    show_annotation_name = FALSE, show_legend = TRUE
  )

  col_fun <- colorRamp2(c(-3, 0, 3), c("#2166AC", "white", "#B2182B"))
  Heatmap(
    mat, name = "log2FC", col = col_fun,
    top_annotation = top_anno, left_annotation = left_anno,
    column_split = factor(period_of_stage[stages_keep],
                          levels = period_order[period_order %in% period_of_stage[stages_keep]]),
    column_gap = unit(3, "mm"),
    row_split = m$block, cluster_rows = TRUE, cluster_row_slices = FALSE,
    cluster_columns = FALSE,
    show_row_names = TRUE, row_names_side = "right",
    show_column_names = TRUE, column_names_rot = 0, column_names_centered = TRUE,
    show_row_dend = TRUE, row_dend_width = unit(10, "mm"),
    row_title = NULL, column_title = NULL,
    row_gap = unit(1.5, "mm"), border = TRUE, na_col = "grey90"
  )
}

sanitize <- function(x) gsub("[^A-Za-z0-9]+", "_", x)

## 追加：带显著性星号版本（`_star` 后缀）

在上面 3 张图的基础上，用 per-stage `padj` 在每个格子叠加星号：
`* padj<0.05`、`** padj<0.01`、`*** padj<0.001`（NA 或不显著留空）。
其他一切与原版一致；原 3 张不改动，新增 3 张 `_star` PDF。

In [ ]:
# ---- 带星号（显著性）版本：仅新增 3 张 _star PDF ----------------------------
stage_padj_cols <- paste0(stage_order, "_padj")

padj_to_star <- function(p) {
  ifelse(is.na(p), "",
    ifelse(p < 0.001, "***",
      ifelse(p < 0.01, "**",
        ifelse(p < 0.05, "*", ""))))
}

make_module_heatmap_star <- function(one_direction, one_ct, one_module) {
  m <- matrix_long %>%
    filter(direction == one_direction, celltype == one_ct, module == one_module) %>%
    distinct(subclass, module, gene, .keep_all = TRUE)
  if (nrow(m) == 0) return(NULL)

  stages_keep <- present_stages(m)
  cols_keep   <- paste0(stages_keep, "_log2FoldChange")
  padj_keep   <- paste0(stages_keep, "_padj")

  m <- m %>%
    mutate(subclass = factor(subclass, levels = fate_order),
           module   = factor(module)) %>%
    arrange(subclass, module)
  m <- m %>% mutate(module = factor(as.character(module), levels = one_module))
  block_chr <- as.character(m$subclass)
  m$block   <- factor(block_chr, levels = intersect(fate_order, unique(block_chr)))

  mat <- as.matrix(m[, cols_keep]);  colnames(mat) <- stages_keep
  rownames(mat) <- make.unique(m$gene)
  star_mat <- matrix(padj_to_star(as.matrix(m[, padj_keep])),
                     nrow = nrow(m), dimnames = list(rownames(mat), stages_keep))

  top_anno <- HeatmapAnnotation(
    `Cell type` = factor(rep(one_ct, length(stages_keep)), levels = one_ct),
    Stage       = factor(stages_keep, levels = stages_keep),
    col = list(`Cell type` = cols.celltype, Stage = cols.stage.full),
    show_annotation_name = FALSE, show_legend = TRUE)
  left_anno <- rowAnnotation(
    Subclass = m$subclass, Module = m$module,
    col = list(Subclass = cols.subclass.short, Module = cols.module),
    show_annotation_name = FALSE, show_legend = TRUE)

  col_fun <- colorRamp2(c(-3, 0, 3), c("#2166AC", "white", "#B2182B"))
  Heatmap(
    mat, name = "log2FC", col = col_fun,
    top_annotation = top_anno, left_annotation = left_anno,
    column_split = factor(period_of_stage[stages_keep],
                          levels = period_order[period_order %in% period_of_stage[stages_keep]]),
    column_gap = unit(3, "mm"),
    row_split = m$block, cluster_rows = TRUE, cluster_row_slices = FALSE,
    cluster_columns = FALSE,
    show_row_names = TRUE, row_names_side = "right",
    show_column_names = TRUE, column_names_rot = 0, column_names_centered = TRUE,
    show_row_dend = TRUE, row_dend_width = unit(10, "mm"),
    row_title = NULL, column_title = NULL,
    row_gap = unit(1.5, "mm"), border = TRUE, na_col = "grey90",
    cell_fun = function(j, i, x, y, width, height, fill) {
      lab <- star_mat[i, j]
      if (!is.na(lab) && nzchar(lab))
        grid.text(lab, x, y, gp = gpar(fontsize = 9, fontface = "bold", col = "black"))
    }
  )
}

n_star <- 0
for (i in seq_len(nrow(combos))) {
  one_direction <- combos$direction[i]; one_ct <- combos$celltype[i]; one_module <- combos$module[i]
  ht <- make_module_heatmap_star(one_direction, one_ct, one_module)
  if (is.null(ht)) { cat("无数据，跳过:", one_direction, one_ct, one_module, "\n"); next }

  m_probe <- matrix_long %>%
    filter(direction == one_direction, celltype == one_ct, module == one_module) %>%
    distinct(subclass, module, gene, .keep_all = TRUE)
  n_rows <- nrow(m_probe)
  hh <- max(3.2, 1.2 + n_rows * 0.28); ww <- 6.5

  pdf_file <- file.path(plot_dir, paste0(
    date_str, "_celltype_direction_module_log2FC_heatmap_",
    one_ct, "_", one_direction, "_", sanitize(one_module), "_star.pdf"))
  pdf(pdf_file, width = ww, height = hh)
  draw(ht, merge_legend = TRUE, heatmap_legend_side = "right", annotation_legend_side = "right")
  dev.off()
  n_star <- n_star + 1
  cat("已输出:", basename(pdf_file), sprintf("(genes=%d)\n", n_rows))
}
cat("\n星号版完成，共输出", n_star, "张 PDF。\n")

# 内联预览（星号版）
library(IRdisplay)
tmp_star_dir <- file.path(tempdir(), "logfc_module_star")
dir.create(tmp_star_dir, showWarnings = FALSE, recursive = TRUE)
for (i in seq_len(nrow(combos))) {
  one_direction <- combos$direction[i]; one_ct <- combos$celltype[i]; one_module <- combos$module[i]
  ht <- make_module_heatmap_star(one_direction, one_ct, one_module)
  if (is.null(ht)) next
  m_probe <- matrix_long %>%
    filter(direction == one_direction, celltype == one_ct, module == one_module) %>%
    distinct(subclass, module, gene, .keep_all = TRUE)
  hh <- max(3.2, 1.2 + nrow(m_probe) * 0.28)
  tmp_png <- file.path(tmp_star_dir, paste0("star_", one_ct, "_", one_direction, "_", sanitize(one_module), ".png"))
  png(tmp_png, width = 6.5 * 120, height = hh * 120, res = 120)
  draw(ht, merge_legend = TRUE, heatmap_legend_side = "right", annotation_legend_side = "right")
  dev.off()
  display_png(file = tmp_png)
}
cat("星号版内联预览完成。\n")